# Iskalna drevesa

## [Preskočni seznami](https://en.wikipedia.org/wiki/Skip_list)

Preskočni seznami so razširitev verižnih seznamov z več nivoji bližnjic (kot izvozi na hitrih cestah), ki omogočajo hitrejše iskanje elementov, na primer:

```markdown
-∞                      →                          ∞
 ↓                                                 ↓
-∞ → 2   →   5          →                18   →    ∞
 ↓   ↓       ↓                           ↓         ↓
-∞ → 2   →   5 → 7      →                18   →    ∞
 ↓   ↓       ↓   ↓                       ↓         ↓
-∞ → 2   →   5 → 7      →      13   →    18 → 20 → ∞
 ↓   ↓       ↓   ↓             ↓         ↓    ↓    ↓
-∞ → 2 → 3 → 5 → 7 → 10 → 11 → 13 → 15 → 18 → 20 → ∞
```

Na začetku in koncu vsakega nivoja imamo _stražarja_ (_sentinel_), ki nam olajša implementacijo robnih pogojev.

- **iskanje**: Iskati začnemo na najvišjem nivoju in se premikamo desno, dokler ne najdemo elementa ali presežemo cilj. Če presežemo cilj, se korak prej spustimo na nižji nivo in iskanje nadaljujemo tam.
- **vstavljanje**: Najprej poiščemo pravo mesto za vstavljanje, nato pa zaporedoma mečemo kovanec (običajno z verjetnostjo $1/2$), da določimo, koliko časa element še dodajamo na višje nivoje.
- **brisanje**: Najprej poiščemo element, nato pa ga s prevezavo kazalcev odstranimo iz vseh nivojev, na katerih se pojavi.
- z malo truda lahko na desnih povezavah beležimo tudi dolžino bližnjic, kar nam omogoča hitro **indeksiranje** oz. dostopanje do elementov glede na njihov položaj v seznamu.
 
Preskočni seznami omogočajo časovno kompleksnost $O(\log n)$ za iskanje, vstavljanje in brisanje elementov ter so v imperativnih jezikih v nekaterih vidikih preprostejši za implementacijo od uravnoteženih iskalnih dreves.

In [ ]:
type node = {
  value: int;
  mutable right: node option;
  mutable down: node option;
}

In [ ]:
#use "topfind";;
#require "jupyter.notebook"

let to_html start_node =
  (* Traverse down to get all layer head nodes *)
  let rec get_all_rows node acc =
    let acc' = node :: acc in
    match node.down with
    | Some d -> get_all_rows d acc'
    | None -> List.rev acc'
  in
  let rows = get_all_rows start_node [] in
  
  (* The bottom row head is the last in our list; it contains all elements *)
  let bottom_row_head = List.hd (List.rev rows) in
  
  (* Traverse the bottom row to get our "universe" of columns *)
  let rec get_all_values node acc =
    let acc' = node.value :: acc in
    match node.right with
    | Some r -> get_all_values r acc'
    | None -> List.rev acc'
  in
  let all_vals = get_all_values bottom_row_head [] in
  
  (* Helper to format values (handles int limits as sentinels visually) *)
  let format_val v =
    if v = min_int then "-&infin;"
    else if v = max_int then "&infin;"
    else string_of_int v
  in
  
  (* Build a single row corresponding to a level *)
  let render_row head_node =
    let rec build_cells vals current_opt acc =
      match vals with
      | [] -> List.rev acc
      | v :: vs ->
          match current_opt with
          | Some current when current.value = v ->
              let cell = Printf.sprintf 
                "<td style=\"border: 1px solid #ddd; padding: 4px 12px; font-weight: bold;\">%s</td>" 
                (format_val v) 
              in
              build_cells vs current.right (cell :: acc)
          | _ ->
              (* Node skipped on this level *)
              let cell = "<td style=\"border: 1px solid #ddd; padding: 4px 12px; color: lightgray;\">&rarr;</td>" in
              build_cells vs current_opt (cell :: acc)
    in
    "<tr>" ^ String.concat "" (build_cells all_vals (Some head_node) []) ^ "</tr>"
  in
  
  let table_content = String.concat "\n" (List.map render_row rows) in
  Printf.sprintf "<table style=\"border-collapse: collapse; text-align: center; font-family: monospace;\">\n%s\n</table>" table_content

let display_skip_list node =
  Jupyter_notebook.printf "%s" (to_html node);
  Jupyter_notebook.display_formatter "text/html"

In [ ]:
let rec new_list levels =
  if levels = 0 then None
  else Some { value = min_int; right = None; down = new_list (levels - 1) }

In [ ]:
let get_path node x =
  let rec aux acc =
    function
    | { right = Some r; _ } when r.value < x -> 
        aux acc r
    | current ->
        let acc' = current :: acc in
        match current.down with
        | Some d -> aux acc' d
        | None -> acc'
  in
  aux [] node

In [ ]:
let find node x =
  match get_path node x with
  | { right = Some { value = y; _ }; _ } :: _ -> x = y
  | _ -> false

In [ ]:
let insert node x =
  let path = get_path node x in
  match path with
  | [] -> assert false
  | { right = Some { value = y; _ }; _ } :: _ when y = x -> () (* Element already exists, do nothing *)
  | path ->
      let rec amend_path previous_new_node =
        function
        | [] -> ()
        | pred :: rest ->
            let new_node = Some { value = x; right = pred.right; down = previous_new_node } in
            pred.right <- new_node;
            if Random.bool () then amend_path new_node rest
      in
      amend_path None path

In [ ]:
let a = new_list 5 |> Option.get

In [ ]:
List.iter (insert a) [2; 3; 5; 8; 12; 21; 34; 55; 89; 144; 233; 377; 610; 987; 1597; 2584; 4181; 6765]

In [ ]:
display_skip_list a

**Trditev**
Naj bo $n$ število elementov v preskočnem seznamu. Potem velja:
- Pričakovano število nivojev je $O(\log n)$.
- Pričakovano število vozlišč je $2 n + O(\log n)$.
- Pričakovana dolžina poti od začetka do iskanega elementa je $O(\log n)$.

_Dokaz_:

Verjetnost, da se element pojavi na nivoju $L_i$ je $1/2^i$, zato je $E[|L_i|] = n / 2^i$. Očitno je $[L_i \ne \emptyset] \le |L_i|$, zato je $E[[L_i \ne \emptyset]] \le E[|L_i|] = n / 2^i$. Skupno pričakovano število nivojev je torej

$$
    E\Big[\sum_{i=0}^{\infty} [L_i \ne \emptyset]\Big]
    = \sum_{i=0}^{\lfloor \log n \rfloor} E[[L_i \ne \emptyset]] + \sum_{i=\lfloor \log n \rfloor + 1}^{\infty} E[[L_i \ne \emptyset]]
    \le \lfloor 1 + \log n \rfloor \cdot 1 + \sum_{i=0}^{\infty} n / 2^{\lfloor \log n \rfloor + i} = \log n + 2
$$

Pričakovano število notranjih vozlišč je torej $\sum_{i=0}^{\infty} E[|L_i|] = \sum_{i=0}^{\infty} n / 2^i = 2 n$, k čimer moramo prišteti še $O(\log n)$ vozlišč za stražarje.

Dolžina poti od začetka do iskanega elementa je enaka dolžini obratne poti od elementa do začetka, kjer se premaknemo navzgor, če se lahko, sicer se premaknemo levo. Verjetnost za korak navzgor je $1/2$, bomo naredili $O(\log n)$ korakov navzgor. Ker je tudi verjetnost za korak na levo enaka $1 - 1/2 = 1/2$, bomo naredili $O(\log n)$ korakov levo. Na koncu naredimo še $O(\log n)$ korakov navzgor.

## Naključna uravnotežena drevesa (treap)

Združitev drevesa in kopice se v angleščini imenuje **treap** (tree + heap). Vsako vozlišče vsebuje ključ in prioriteto, treap pa je zgrajen tako, da je dvojiški iskalno drevo za ključe, za prioritete pa je kopica. S tem je struktura drevesa natanko določena. Dobimo jo lahko na dva načina:

- Za koren vzamemo element z največjo prioriteto, nato pa rekurzivno zgradimo levo in desno poddrevo iz elementov, ki so manjši oziroma večji od ključa korena.
- V prazno iskalno drevo vstavimo vse ključe, urejene glede na njihove prioritete.

Dodajanje elementov izvedemo enako, kot bi jih v navadno BST drevo, nato pa z levimi in desnimi rotacijami element splavamo navzgor, dokler ne zadostimo pogoju kopice. Brisanje deluje v obratni smeri: vozlišče z rotacijami potopimo proti otroku z večjo prioriteto. Ko pridemo do dna, je brisanje trivialno.

Na ta način lahko gradimo drevesa, v katerih so pogosteje uporabljani elementi bližje korenu. Bolj pogosta uporaba treapov pa v implementaciji izbere naključno prioriteto, kar zagotavlja pričakovano dobro uravnoteženost ne glede na vrstni red vstavljanja elementov.

In [ ]:
type 'a treap =
  | Empty
  | Node of {
      key: 'a;
      priority: float;
      left: 'a treap;
      right: 'a treap;
    }

(* Rotacija v desno povzdigne levo drevo *)
let rotate_right = function
  | Node { key = y; priority = py; left = Node { key = x; priority = px; left = a; right = b }; right = c } ->
      Node { key = x; priority = px; left = a; right = Node { key = y; priority = py; left = b; right = c } }
  | _ -> assert false

(* Rotacija v levo povzdigne desno drevo *)
let rotate_left = function
  | Node { key = x; priority = px; left = a; right = Node { key = y; priority = py; left = b; right = c } } ->
      Node { key = y; priority = py; left = Node { key = x; priority = px; left = a; right = b }; right = c }
  | _ -> assert false

let rec find k = function
  | Empty -> false
  | Node { key; left; right; _ } ->
      if k = key then true
      else if k < key then find k left
      else find k right

(* Dodajanje elementa uporabi klasično bst dodajanje in nato rotacije ob vračanju, 
   če je bila porušena pravilo kopice (max-heap) *)
let rec insert k = function
  | Empty -> Node { key = k; priority = Random.float 1.0; left = Empty; right = Empty }
  | Node n ->
      if k = n.key then Node n (* Element že obstaja *)
      else if k < n.key then
        let left' = insert k n.left in
        let root' = Node { n with left = left' } in
        match left' with
        | Node ln when ln.priority > n.priority -> rotate_right root'
        | _ -> root'
      else
        let right' = insert k n.right in
        let root' = Node { n with right = right' } in
        match right' with
        | Node rn when rn.priority > n.priority -> rotate_left root'
        | _ -> root'

(* Brisanje elementa uporabi rotacije, da potisne vozlišče navzdol (proti listom), 
   kjer ga preprosto odstranimo *)
let rec delete k = function
  | Empty -> Empty
  | Node n as t ->
      if k < n.key then Node { n with left = delete k n.left }
      else if k > n.key then Node { n with right = delete k n.right }
      else
        (* Našli smo vozlišče, ki ga želimo izbrisati. *)
        match n.left, n.right with
        | Empty, r -> r
        | l, Empty -> l
        | Node ln, Node rn ->
            (* Potisnemo vozlišče navzdol po tisti poti, kjer ima otrok večjo prioriteto *)
            if ln.priority > rn.priority then
              let t' = rotate_right t in
              match t' with
              | Node n' -> Node { n' with right = delete k n'.right }
              | Empty -> assert false
            else
              let t' = rotate_left t in
              match t' with
              | Node n' -> Node { n' with left = delete k n'.left }
              | Empty -> assert false

In [ ]:
(* Pomožna funkcija za generiranje Graphviz DOT niza iz našega treapa *)
let treap_to_dot treap =
  let buf = Buffer.create 1024 in
  let push = Buffer.add_string buf in
  let null_count = ref 0 in
  let get_null () =
    incr null_count;
    Printf.sprintf "null%d" !null_count
  in
  let rec aux = function
    | Empty -> ()
    | Node { key; priority; left; right } ->
        (* Polje tipa record uporabimo za prikaz tako ključa kot prioritete *)
        push (Printf.sprintf "  %d [label=<%d<SUB>%f</SUB>>];\n" key key priority);
        (match left, right with
        | Empty, Empty -> ()
        | Node l, Empty -> 
            push (Printf.sprintf "  %d -> %d;\n" key l.key);
            (* Dodamo neviden naključen list, da bo levo poddrevo zares narisano levo *)
            let n = get_null () in
            push (Printf.sprintf "  %s [shape=point, style=invis];\n" n);
            push (Printf.sprintf "  %d -> %s [style=invis];\n" key n);
            aux left
        | Empty, Node r ->
            let n = get_null () in
            push (Printf.sprintf "  %s [shape=point, style=invis];\n" n);
            push (Printf.sprintf "  %d -> %s [style=invis];\n" key n);
            push (Printf.sprintf "  %d -> %d;\n" key r.key);
            aux right
        | Node l, Node r ->
            push (Printf.sprintf "  %d -> %d;\n" key l.key);
            push (Printf.sprintf "  %d -> %d;\n" key r.key);
            aux left;
            aux right)
  in
  push "digraph Treap {\n  node [style=rounded];\n";
  aux treap;
  push "}\n";
  Buffer.contents buf

let narisi drevo =
  let ime_datoteke = "drevo.dot" in
  let datoteka = open_out ime_datoteke in
  let dot = treap_to_dot drevo in
  Printf.fprintf datoteka "%s\n" dot;
  close_out datoteka;
  ignore (Jupyter_notebook.Process.sh "dot -Tsvg drevo.dot > drevo.svg");
  ignore (Jupyter_notebook.display_file "image/svg+xml" "drevo.svg")


In [ ]:
List.fold_right insert (List.init 20 (fun i -> i)) Empty |> narisi

**NE PRIDE V POŠTEV ZA IZPIT**

Naj bodo $x_1, x_2, \dots, x_n$ ključi in $p_1, p_2, \dots, p_n$ njihove prioritete. Definirajmo $i \uparrow k$, kadar je $x_i$ v drevesu predhodnik $x_k$.

Pokažimo, da je pričakovana višina vozlišča $O(\log n)$. Pričakovana višina vozlišča $x_k$ je $E[\sum_{i=1}^n [i \uparrow k]]$. Oglejmo si primer, ko je $i < k$, saj je primer $i > k$ simetričen. Pokažimo, da $i \uparrow k$ velja natanko tedaj, kadar je $i = \argmax_{i \le j \le k} p_j$, tj. kadar ima $i$ največjo prioriteto med vozlišči $\{ x_i, x_{i+1}, \dots, x_k \}$. Ločimo štiri primere:

- če je $x_i$ koren velja tako $i \uparrow k$ kot to, da je $p_i$ največja prioriteta;
- če je $x_k$ koren ne velja niti $i \uparrow k$ niti to, da je $p_i$ največja prioriteta;
- če je koren neko tretje vozlišče $x_j$, imamo dva podprimera:
  -  če sta $x_i$ in $x_k$ v različnih poddrevesih $x_j$, potem je tudi $i \le j \le k$, zato zopet ne velja nobena od trditev;
  - če sta $x_i$ in $x_k$ v istem poddrevesu, trditev velja po indukciji.

Ker so prioritete naključne, je verjetnost, da ima $i$ največjo prioriteto v $\{ x_i, x_{i+1}, \dots, x_k \}$, enaka $1 / (k - i + 1)$.

Torej je

$$\begin{align*}
    E\Big[\sum_{i=1}^n [i \uparrow k]\Big]
    &= \sum_{i=1}^{k-1} P(i \uparrow k) + \sum_{i=k+1}^n P(i \uparrow k) \\
    &= \sum_{i=1}^{k-1} \frac{1}{k - i + 1} + \sum_{i=k+1}^n \frac{1}{i - k + 1} \\
    &= \sum_{i=2}^{k} \frac{1}{i} + \sum_{i=2}^{n-k+1} \frac{1}{i} \\
    &< \ln k + \ln (n - k + 1) - 2 \\
    &= O(\log n)
\end{align*}$$
